# TFG V4: Cost-phase + local mixer

Nueva version basada en `TFG_V3`. El objetivo es aumentar experimentalmente la probabilidad de medir ventanas validas sin usar Grover directamente y sin postseleccion como mecanismo principal.

La idea es:

1. Preparar una superposicion uniforme sobre indices validos.
2. Cargar reversiblemente la ventana asociada a cada indice.
3. Aplicar una fase dependiente del coste `C(i) = numero de unos en window_i`.
4. Descomputar la ventana para dejar la fase efectiva sobre `idx`.
5. Aplicar un mixer local/modular sobre `idx`.
6. Repetir el bloque varias veces y analizar las probabilidades por indice.

El mixer por defecto no es la difusion global de Grover. Se implementa como rotaciones locales entre indices vecinos validos, definidos por adyacencia geometrica de las coordenadas iniciales de las ventanas.

In [ ]:
import numpy as np
import qiskit

from math import prod
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import MCXGate, UnitaryGate

print(qiskit.__version__)

In [ ]:
# =========================================================
# Parametros configurables del experimento principal
# =========================================================

D = 2
N = [4, 4]          # tamano de la red por dimension
M = [2, 2]          # tamano de la ventana/job por dimension
occupied_coords = [(0, 0), (0, 3), (3, 0), (3, 3)]

# Bloque cost-phase + mixer
theta = np.pi       # fase acumulada por cada 1 de la ventana
mixer_angle = 0.35  # angulo del mixer local
repetitions = 2
use_cost_phase = True
use_mixer = True

# Mixers disponibles:
# - "local_geometric": rota entre ventanas validas vecinas en la geometria ND
# - "linear_valid": rota entre indices validos consecutivos
# - "rx_all": prototipo simple con RX en todos los bits de idx; puede fugar a indices invalidos
mixer_method = "local_geometric"

In [ ]:
# =========================================================
# Geometria ND y utilidades clasicas
# =========================================================

def validate_problem(N, M):
    if len(N) != len(M):
        raise ValueError("N y M deben tener la misma dimension.")
    for d, (n_d, m_d) in enumerate(zip(N, M)):
        if n_d <= 0 or m_d <= 0:
            raise ValueError(f"N[{d}] y M[{d}] deben ser positivos.")
        if m_d > n_d:
            raise ValueError(f"M[{d}] no puede ser mayor que N[{d}].")


def coord_to_index(coord, dims):
    """Convierte coordenadas ND a indice lineal row-major."""
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def index_to_coord(index, dims):
    """Convierte indice lineal row-major a coordenadas ND."""
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)


def valid_starts_nd(N, M):
    """Coordenadas iniciales validas para una ventana M dentro de N."""
    return list(np.ndindex(tuple(N[d] - M[d] + 1 for d in range(len(N)))))


def window_qubits_nd(start, N, M):
    """Indices lineales de las celdas cubiertas por la ventana que empieza en start."""
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def normalize_coord(coord, D):
    if D == 1 and isinstance(coord, int):
        return (coord,)
    return tuple(coord)


def build_grid_bits(N, occupied_coords):
    """Devuelve un vector clasico con 1 en celdas ocupadas y 0 en celdas libres."""
    D = len(N)
    grid = [0] * prod(N)
    for raw_coord in occupied_coords:
        coord = normalize_coord(raw_coord, D)
        if len(coord) != D:
            raise ValueError(f"Coordenada {coord} no tiene dimension {D}.")
        for d, x in enumerate(coord):
            if x < 0 or x >= N[d]:
                raise ValueError(f"Coordenada {coord} fuera de la red N={N}.")
        grid[coord_to_index(coord, N)] = 1
    return grid


def compute_window_cost_classical(grid_bits, start, N, M):
    """C(i): numero de unos en la ventana asociada a start."""
    return sum(grid_bits[q] for q in window_qubits_nd(start, N, M))


def window_string_classical(grid_bits, start, N, M):
    return ''.join(str(grid_bits[q]) for q in window_qubits_nd(start, N, M))


def get_valid_indices(grid_bits, starts, N, M):
    return [i for i, start in enumerate(starts) if compute_window_cost_classical(grid_bits, start, N, M) == 0]


def gray_order_valid(W, IDX):
    gray_full = [t ^ (t >> 1) for t in range(2**IDX)]
    return [g for g in gray_full if g < W]


def format_nd_array_from_bits(bitstring, dims):
    arr = np.array(list(bitstring), dtype=str).reshape(tuple(dims))
    return np.array2string(arr, separator=' ').replace("'", "")

In [ ]:
# =========================================================
# Bloques cuanticos: loader reversible, fase de coste y mixers
# =========================================================

def append_mcx(qc, controls, target):
    """MCX moderno mediante MCXGate, sin usar argumentos deprecated como mode='noancilla'."""
    if len(controls) == 0:
        qc.x(target)
    elif len(controls) == 1:
        qc.cx(controls[0], target)
    else:
        qc.append(MCXGate(num_ctrl_qubits=len(controls)), controls + [target])


def apply_window_loader(qc, n, idx, m, starts, N, M, order_valid):
    """
    Implementa L: |grid>|i>|m> -> |grid>|i>|m xor window_i>.
    Aplicar esta funcion dos veces descomputa la ventana porque el bloque es XOR reversible.
    """
    IDX = len(idx)
    current_zero_mask = [False] * IDX

    for i in order_valid:
        bits = [(i >> b) & 1 for b in range(IDX)]  # little-endian en Qiskit
        target_zero_mask = [bits[b] == 0 for b in range(IDX)]

        for b in range(IDX):
            if current_zero_mask[b] != target_zero_mask[b]:
                qc.x(idx[b])
                current_zero_mask[b] = target_zero_mask[b]

        for j, n_pos in enumerate(window_qubits_nd(starts[i], N, M)):
            controls = [idx[b] for b in range(IDX)] + [n[n_pos]]
            append_mcx(qc, controls, m[j])

    for b in range(IDX):
        if current_zero_mask[b]:
            qc.x(idx[b])


def apply_cost_phase(qc, m, theta):
    """
    Si m contiene window_i, entonces cada qubit m[j]=1 acumula fase exp(i theta).
    El estado |window_i> acumula exp(i theta C(i)).
    """
    for q in m:
        qc.p(theta, q)


def two_level_mixer_gate(num_qubits, a, b, beta, label=None):
    """
    Rotacion local en el subespacio span{|a>, |b>}:
        exp(-i beta X_ab)
    y accion identidad sobre el resto. Es prototipo modular para grafos pequenos.
    """
    dim = 2**num_qubits
    U = np.eye(dim, dtype=complex)
    c = np.cos(beta)
    s = -1j * np.sin(beta)
    U[a, a] = c
    U[b, b] = c
    U[a, b] = s
    U[b, a] = s
    return UnitaryGate(U, label=label or f"Mix({a},{b})")


def mixer_edges_from_starts(starts, N, method="local_geometric"):
    """Aristas locales entre indices validos. No implementa difusion global de Grover."""
    if method == "linear_valid":
        return [(i, i + 1) for i in range(len(starts) - 1)]

    if method != "local_geometric":
        raise ValueError("Metodo de mixer local desconocido.")

    start_to_idx = {tuple(s): i for i, s in enumerate(starts)}
    edges = []
    D = len(N)
    for i, start in enumerate(starts):
        for d in range(D):
            neigh = list(start)
            neigh[d] += 1
            neigh = tuple(neigh)
            if neigh in start_to_idx:
                edges.append((i, start_to_idx[neigh]))
    return edges


def apply_mixer(qc, idx, starts, N, mixer_angle, method="local_geometric"):
    """
    Mixer modular sobre el registro idx.
    - local_geometric: mezcla ventanas vecinas en la geometria de starts.
    - linear_valid: mezcla indices consecutivos validos.
    - rx_all: prototipo simple; puede transferir amplitud a indices invalidos si W no es potencia de 2.
    """
    if abs(mixer_angle) < 1e-15:
        return

    if method == "rx_all":
        for q in idx:
            qc.rx(2 * mixer_angle, q)
        return

    IDX = len(idx)
    for a, b in mixer_edges_from_starts(starts, N, method):
        qc.append(two_level_mixer_gate(IDX, a, b, mixer_angle), list(idx))

In [ ]:
# =========================================================
# Construccion del circuito y analisis de probabilidades
# =========================================================

def prepare_valid_index_superposition(qc, idx, W):
    IDX = len(idx)
    amps = np.zeros(2**IDX, dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


def prepare_grid_register(qc, n, N, occupied_coords):
    for q in [coord_to_index(normalize_coord(c, len(N)), N) for c in occupied_coords]:
        qc.x(n[q])


def build_cost_phase_mixer_circuit(
    N, M, occupied_coords, theta, mixer_angle, repetitions,
    use_cost_phase=True, use_mixer=True, mixer_method="local_geometric",
    add_barriers=True,
):
    validate_problem(N, M)
    D = len(N)
    N_tot = prod(N)
    M_tot = prod(M)
    starts = valid_starts_nd(N, M)
    W = len(starts)
    IDX = int(np.ceil(np.log2(W))) if W > 1 else 1
    order_valid = gray_order_valid(W, IDX)
    grid_bits = build_grid_bits(N, occupied_coords)

    n = QuantumRegister(N_tot, "n")
    idx = QuantumRegister(IDX, "i")
    m = QuantumRegister(M_tot, "m")
    qc = QuantumCircuit(n, idx, m)

    prepare_grid_register(qc, n, N, occupied_coords)
    prepare_valid_index_superposition(qc, idx, W)
    if add_barriers:
        qc.barrier()

    for r in range(repetitions):
        apply_window_loader(qc, n, idx, m, starts, N, M, order_valid)
        if use_cost_phase:
            apply_cost_phase(qc, m, theta)
        apply_window_loader(qc, n, idx, m, starts, N, M, order_valid)
        if use_mixer:
            apply_mixer(qc, idx, starts, N, mixer_angle, mixer_method)
        if add_barriers:
            qc.barrier()  # end of cost-phase + mixer layer

    metadata = {
        "D": D, "N": list(N), "M": list(M), "N_tot": N_tot, "M_tot": M_tot,
        "starts": starts, "W": W, "IDX": IDX, "grid_bits": grid_bits,
        "occupied_coords": [normalize_coord(c, D) for c in occupied_coords],
        "theta": theta, "mixer_angle": mixer_angle, "repetitions": repetitions,
        "mixer_method": mixer_method,
    }
    return qc, metadata


def index_probabilities_from_statevector(sv, metadata):
    N_tot = metadata["N_tot"]
    IDX = metadata["IDX"]
    probs = np.zeros(2**IDX, dtype=float)
    idx_mask = (1 << IDX) - 1
    for basis_idx, amp in enumerate(sv.data):
        prob = float(abs(amp) ** 2)
        if prob == 0.0:
            continue
        idx_int = (basis_idx >> N_tot) & idx_mask
        probs[idx_int] += prob
    return probs


def analyze_probabilities(sv, metadata, title="Analysis", tol=1e-12):
    N, M = metadata["N"], metadata["M"]
    starts = metadata["starts"]
    W = metadata["W"]
    grid_bits = metadata["grid_bits"]
    probs = index_probabilities_from_statevector(sv, metadata)
    valid_indices = get_valid_indices(grid_bits, starts, N, M)

    p_valid_initial = len(valid_indices) / W
    p_valid_after = float(sum(probs[i] for i in valid_indices))
    p_invalid_after = 1.0 - p_valid_after
    p_invalid_index = float(sum(probs[i] for i in range(W, len(probs))))

    print(f"\n============ {title} ============")
    print(f"N={N}, M={M}, W={W}, IDX={metadata['IDX']}")
    print(f"theta={metadata['theta']:.6g}, mixer_angle={metadata['mixer_angle']:.6g}, repetitions={metadata['repetitions']}")
    print(f"mixer_method={metadata['mixer_method']}")
    print(f"valid_indices={valid_indices}")
    print(f"P_valid_initial = {p_valid_initial:.6f}")
    print(f"P_valid_after   = {p_valid_after:.6f}")
    print(f"P_invalid_after = {p_invalid_after:.6f}")
    if p_invalid_index > tol:
        print(f"P_invalid_index_states = {p_invalid_index:.6f}")

    print("\nindex | start coordinate | window | C(i) | probability | valid")
    print("------|------------------|--------|------|-------------|------")
    rows = []
    for i, start in enumerate(starts):
        window = window_string_classical(grid_bits, start, N, M)
        cost = compute_window_cost_classical(grid_bits, start, N, M)
        rows.append((i, start, window, cost, probs[i]))
        print(f"{i:5d} | {str(start):16s} | {window:6s} | {cost:4d} | {probs[i]:11.6f} | {cost == 0}")

    return {
        "probs": probs,
        "valid_indices": valid_indices,
        "P_valid_initial": p_valid_initial,
        "P_valid_after": p_valid_after,
        "P_invalid_after": p_invalid_after,
        "P_invalid_index_states": p_invalid_index,
        "rows": rows,
    }


def run_experiment(name, N, M, occupied_coords, theta=np.pi, mixer_angle=0.35, repetitions=2,
                   mixer_method="local_geometric", draw=False):
    print(f"\n\n########################################")
    print(f"Experiment: {name}")
    print(f"########################################")
    qc, metadata = build_cost_phase_mixer_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, mixer_angle=mixer_angle, repetitions=repetitions,
        use_cost_phase=use_cost_phase, use_mixer=use_mixer,
        mixer_method=mixer_method,
    )
    sv = Statevector(qc)
    analysis = analyze_probabilities(sv, metadata, title=name)
    if draw:
        display(qc.draw(output="mpl"))
    return qc, metadata, analysis

In [ ]:
# =========================================================
# Experimento principal
# =========================================================

qc_v4, meta_v4, analysis_v4 = run_experiment(
    name="Main V4 cost-phase + local mixer",
    N=N,
    M=M,
    occupied_coords=occupied_coords,
    theta=theta,
    mixer_angle=mixer_angle,
    repetitions=repetitions,
    mixer_method=mixer_method,
    draw=False,
)

qc_v4.draw(output="mpl")

## Tests pequenos

Estos tests no son pruebas unitarias formales: son casos pequenos para inspeccionar si la probabilidad valida aumenta o disminuye al variar `theta`, `mixer_angle`, `repetitions` y `mixer_method`. Si algun caso no mejora `P_valid`, no implica fallo del circuito; indica que los parametros o el mixer no son adecuados para esa instancia.

In [ ]:
# Test 1: caso 1D con bloque ocupado consecutivo
qc_test_1d, meta_test_1d, analysis_test_1d = run_experiment(
    name="Test 1D: bloque ocupado consecutivo",
    N=[8],
    M=[2],
    occupied_coords=[(3,), (4,)],
    theta=np.pi,
    mixer_angle=0.30,
    repetitions=2,
    mixer_method="linear_valid",
)

# Test 2: caso 2D pequeno, red 4x4 con ventana 2x2
qc_test_2d, meta_test_2d, analysis_test_2d = run_experiment(
    name="Test 2D: 4x4 con ventana 2x2",
    N=[4, 4],
    M=[2, 2],
    occupied_coords=[(1, 1), (2, 2)],
    theta=np.pi,
    mixer_angle=0.25,
    repetitions=2,
    mixer_method="local_geometric",
)

# Test 3: varias ventanas validas
qc_test_multi, meta_test_multi, analysis_test_multi = run_experiment(
    name="Test 2D: varias ventanas validas",
    N=[4, 4],
    M=[2, 2],
    occupied_coords=[(0, 0), (3, 3)],
    theta=np.pi / 2,
    mixer_angle=0.40,
    repetitions=3,
    mixer_method="local_geometric",
)

## Observaciones de diseno

- La fase de coste se implementa como `P(theta)` sobre cada qubit del registro `m` despues de cargar `window_i`. Esto implementa `exp(i theta C(i))` porque cada celda ocupada contribuye una fase `theta`.
- Despues se aplica el mismo loader para descomputar `m`, dejando la fase acumulada sobre `idx`.
- El mixer por defecto es local sobre el grafo de ventanas validas. No usa el operador de difusion global de Grover.
- `rx_all` queda como prototipo simple, pero puede transferir amplitud a indices invalidos cuando `W` no es potencia de dos.
- Para instancias grandes, el mixer local basado en `UnitaryGate` deberia sustituirse por una descomposicion mas eficiente en puertas nativas o por mixers geometricos especializados.

## Requisitos de ejecucion

Para ejecutar este notebook hace falta tener instaladas las dependencias del proyecto. Desde la carpeta `TFG`:

```bash
pip install -r requirements.txt
```

En particular, se requiere `qiskit`, `qiskit-aer` y `numpy`. Si aparece `ModuleNotFoundError: No module named 'qiskit'`, el problema es de entorno y no de sintaxis del notebook.